# Proposed validation — review before it runs

**What this measures:** abs_diff_total_loss is the max absolute difference between lightly's ported MCRLoss.forward output and an embedded line-for-line port of the SimDINO reference MCRLoss, across configurations (expa_type 0/1 and a non-default eps/coeff pair) on the fixed seed-42 input — it directly measures whether the port reproduces the reference's numerics, exactly as the claim states.

**Target metric:** `abs_diff_total_loss`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

The executable copy lives at `eval/eval_mcr_loss_parity.py`, which is what `.remyx/validation.yaml` points at; keep the two in step, or point `suite:` here if you would rather maintain the notebook.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the script at `eval/eval_mcr_loss_parity.py`, unchanged. This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_mcr_loss_parity.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
#!/usr/bin/env python
"""Parity eval: lightly MCRLoss vs an embedded reference SimDINO MCRLoss port.

Measures numerical agreement between the ported lightly.loss.MCRLoss and a
line-for-line port of the author's reference implementation
(RobinWu218/SimDINO), on a fixed seeded input, per VALIDATION.md /
tests/loss/test_mcr_loss.py::TestMCRLossParity. Also checks that gradients
flow (finite) to both student and teacher leaves, since SimDINO does not
detach the teacher.

baseline: none — MCRLoss does not exist pre-change; the parity comparison
lives entirely inside the PR-head arm (lightly impl vs embedded reference).
On the baseline arm the import fails and this script falls back to a
degraded/sentinel parity metric (crash-avoidance, per item 2). The
gradient-flow guardrail, however, uses a trivial MCRLoss-independent
computation on BOTH arms so that it is genuinely achievable pre-change (it
only fails on a real autograd/NaN regression, not on the mere absence of
MCRLoss).
"""
import json
import os
import sys

In [ ]:
import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch import distributed as torch_dist
from torch.distributed import nn as torch_dist_nn

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

In [ ]:
try:
    from lightly.loss import MCRLoss
except (ImportError, AttributeError):
    MCRLoss = None

def l2_normalize(features: Tensor) -> Tensor:
    return F.normalize(features, dim=-1, p=2)

In [ ]:
def random_views(num_views: int, batch_size: int, feature_dim: int):
    return [l2_normalize(torch.randn(batch_size, feature_dim)) for _ in range(num_views)]

def seeded_inputs(n_teacher: int, n_student: int, batch_size: int, feature_dim: int):
    torch.manual_seed(42)
    teacher_out = random_views(n_teacher, batch_size, feature_dim)
    student_out = random_views(n_student, batch_size, feature_dim)
    return teacher_out, student_out

In [ ]:
class ReferenceMCRLoss(nn.Module):
    """Port of the reference SimDINO MCRLoss (MIT, (c) 2025 Ziyang Wu)."""

    def __init__(self, ncrops, reduce_cov=0, expa_type=0, eps=0.5, coeff=1.0):
        super().__init__()
        self.ncrops = ncrops
        self.eps = eps
        self.coeff = coeff
        self.reduce_cov = reduce_cov
        self.expa_type = expa_type

    def forward(self, student_feat, teacher_feat):
        student_feat = student_feat.view(self.ncrops, -1, student_feat.shape[-1])
        teacher_feat = teacher_feat.view(2, -1, teacher_feat.shape[-1])
        comp_loss = self.calc_compression(student_feat, teacher_feat)
        if self.expa_type == 0:
            expa_loss = self.calc_expansion(student_feat[: len(teacher_feat)])
        else:
            expa_loss = self.calc_expansion((student_feat[: len(teacher_feat)] + teacher_feat) / 2)
        loss = -self.coeff * comp_loss - expa_loss
        return loss, comp_loss.detach(), expa_loss.detach()

    def calc_compression(self, student_feat_list, teacher_feat_list):
        sim = F.cosine_similarity(teacher_feat_list.unsqueeze(1), student_feat_list.unsqueeze(0), dim=-1)
        diag = min(len(teacher_feat_list), len(student_feat_list))
        sim[torch.arange(diag), torch.arange(diag)] = 0
        n_loss_terms = len(teacher_feat_list) * len(student_feat_list) - min(
            len(teacher_feat_list), len(student_feat_list)
        )
        return sim.mean(2).sum() / n_loss_terms

    def calc_expansion(self, feat_list):
        num_views = len(feat_list)
        m, p = feat_list[0].shape
        cov_list = torch.stack([W.T.matmul(W) for W in feat_list])
        N = 1
        if torch_dist.is_initialized():
            N = torch_dist.get_world_size()
            if self.reduce_cov == 1:
                cov_list = torch_dist_nn.all_reduce(cov_list)
        scalar = p / (m * N * self.eps)
        I = torch.eye(p, device=cov_list[0].device)
        loss = cov_list.new_zeros(())
        for i in range(num_views):
            chol = torch.linalg.cholesky_ex(I + scalar * cov_list[i])[0]
            loss += chol.diagonal().log().sum()
        loss /= num_views
        loss *= (p + N * m) / (p * N * m)
        return loss

In [ ]:
SMOKE = os.environ.get("REMYX_SMOKE") == "1"
BATCH_SIZE = 4 if SMOKE else 8
FEATURE_DIM = 4 if SMOKE else 16
N_TEACHER, N_STUDENT = 2, (3 if SMOKE else 6)

CONFIGS = [dict(expa_type=0, eps=0.5, coeff=1.0)]
if not SMOKE:
    CONFIGS += [dict(expa_type=1, eps=0.5, coeff=1.0), dict(expa_type=0, eps=0.3, coeff=2.0)]

In [ ]:
def grad_flow_check(compute_loss_fn):
    """Backward through compute_loss_fn(teacher_leaves, student_leaves) and
    return the fraction of the (teacher + student) leaves that receive a
    finite gradient.

    ``compute_loss_fn`` is supplied by the caller so this helper is
    baseline-safe: on the baseline arm it is fed a trivial computation that
    does not depend on the (not-yet-existing) MCRLoss, so the guardrail is
    genuinely achievable pre-change and only fails on a real autograd / NaN
    regression, not on the mere absence of the new class.
    """
    teacher_out, student_out = seeded_inputs(N_TEACHER, N_STUDENT, BATCH_SIZE, FEATURE_DIM)
    teacher_leaves = [t.clone().detach().requires_grad_(True) for t in teacher_out]
    student_leaves = [s.clone().detach().requires_grad_(True) for s in student_out]
    loss = compute_loss_fn(
        [l2_normalize(t) for t in teacher_leaves],
        [l2_normalize(s) for s in student_leaves],
    )
    loss.backward()
    leaves = teacher_leaves + student_leaves
    finite_leaves = sum(
        1 for t in leaves if t.grad is not None and torch.isfinite(t.grad).all().item()
    )
    return finite_leaves / len(leaves)

In [ ]:
if MCRLoss is None:
    # Pre-change baseline: MCRLoss does not exist yet, so parity against the
    # embedded reference cannot be computed and the (target) parity metric
    # legitimately degrades to a sentinel value here. The gradient-flow
    # guardrail below is independent of MCRLoss (basic sum-of-squares),
    # so it is trivially satisfiable on this arm too.
    def _fallback_loss(teacher_feats, student_feats):
        return sum((t**2).sum() for t in teacher_feats) + sum(
            (s**2).sum() for s in student_feats
        )

    grad_finite_fraction = grad_flow_check(_fallback_loss)
    metrics = {
        "abs_diff_total_loss": 1.0,
        "abs_diff_comp_loss": 1.0,
        "abs_diff_expa_loss": 1.0,
        "grad_finite_fraction": grad_finite_fraction,
    }
else:
    max_diff_total = max_diff_comp = max_diff_expa = 0.0
    for cfg in CONFIGS:
        teacher_out, student_out = seeded_inputs(N_TEACHER, N_STUDENT, BATCH_SIZE, FEATURE_DIM)
        loss_fn = MCRLoss(eps=cfg["eps"], coeff=cfg["coeff"], expa_type=cfg["expa_type"])
        total = loss_fn(teacher_out=teacher_out, student_out=student_out)
        comp = loss_fn.calc_compression(teacher_feat_list=teacher_out, student_feat_list=student_out)
        if cfg["expa_type"] == 0:
            expa_features = student_out[: len(teacher_out)]
        else:
            expa_features = [(s + t) / 2 for s, t in zip(student_out[: len(teacher_out)], teacher_out)]
        expa = loss_fn.calc_expansion(feat_list=expa_features)

        ref = ReferenceMCRLoss(ncrops=len(student_out), expa_type=cfg["expa_type"], eps=cfg["eps"], coeff=cfg["coeff"])
        ref_total, ref_comp, ref_expa = ref(torch.cat(student_out), torch.cat(teacher_out))

        max_diff_total = max(max_diff_total, (total - ref_total).abs().item())
        max_diff_comp = max(max_diff_comp, (comp - ref_comp).abs().item())
        max_diff_expa = max(max_diff_expa, (expa - ref_expa).abs().item())

    # Gradient-flow check: SimDINO does not detach the teacher, unlike DINOLoss.
    grad_finite_fraction = grad_flow_check(
        lambda teacher_feats, student_feats: MCRLoss()(
            teacher_out=teacher_feats, student_out=student_feats
        )
    )

    metrics = {
        "abs_diff_total_loss": max_diff_total,
        "abs_diff_comp_loss": max_diff_comp,
        "abs_diff_expa_loss": max_diff_expa,
        "grad_finite_fraction": grad_finite_fraction,
    }

print(json.dumps(metrics))

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
loop: {max_iterations: 8, fix_code: true}
benchmarks:
  - name: "mcr-loss-parity"
    suite: "eval/eval_mcr_loss_parity.py"
    packages: []
    scorer: abs_diff_total_loss
    baseline: none
    metrics:
      - name: abs_diff_total_loss
        role: target
        direction: min
        threshold: 1.0e-06
      - name: grad_finite_fraction
        role: guardrail
        direction: max
        threshold: 1.0
      - name: abs_diff_comp_loss
        direction: min
        threshold: 1.0e-06
      - name: abs_diff_expa_loss
        direction: min
        threshold: 1.0e-06
    policy: {guardrail_veto: true}
    compute: {tier: cpu}
    held_constant:
      - "torch.manual_seed(42) fixed input, teacher shape 2x8x16, student shape 6x8x16, float32"
      - "single-process execution, no torch.distributed init (reduce_cov guard is a no-op)"
      - "reference implementation is the embedded ReferenceMCRLoss port from tests/loss/test_mcr_loss.py, not reimplemented inline"
      - "the grad_finite_fraction guardrail uses a trivial MCRLoss-independent computation on the baseline arm so it is achievable pre-change; it only tests real autograd/NaN behavior on the PR-head arm"
    avoid:
      - "no wall-clock timing; this is a pure numerics parity check"
      - "no unpinned torch build differences across arms; same interpreter/torch version used for the single measured arm"
    provenance:
      abs_diff_total_loss: "user_guidance"
      abs_diff_comp_loss: "repo_runner:tests/loss/test_mcr_loss.py::TestMCRLossParity"
      abs_diff_expa_loss: "repo_runner:tests/loss/test_mcr_loss.py::TestMCRLossParity"
      grad_finite_fraction: "maintainer_comment:VALIDATION.md (gradient flow to student and teacher)"
      held_constant: "repo_runner:tests/loss/test_mcr_loss.py::TestMCRLossParity._seeded_inputs"
      baseline: "claim_analysis:candidate_set (comparison lives inside the PR-head arm against an embedded reference port; MCRLoss does not exist pre-change; only the gradient-flow guardrail is scored on the baseline arm, via an MCRLoss-independent computation)"
question:
  kind: capability
  ask: "The ported lightly MCRLoss (SimDINO coding-rate loss) reproduces the same compression, expansion, and total loss values as the author's reference MCRLoss implementation on a fixed, seeded input."
```